<a href="https://colab.research.google.com/github/salphonseds/llm-from-scratch/blob/main/notebooks/11_Instruction_Tuning_SFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — Setup: GPT-2 small + the NB09/NB10 harness, verbatim
#
# NB11: Instruction Tuning & SFT (Days 19-20)
# Measurement discipline: perplexity(), the tokenizer, and the WikiText
# pipeline (including the "\n\n" join) are byte-identical to NB09/NB10,
# so general PPL remains one unbroken yardstick across all three
# fine-tuning notebooks. Domain axis (Shakespeare) retires this notebook.
!pip install -q transformers datasets

import torch, gc, math, time, random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {torch.cuda.get_device_name(0)}")
torch.manual_seed(42); np.random.seed(42); random.seed(42)

# ---- idempotent state reset ----
for _name in ["model", "optimizer", "scaler"]:
    if _name in globals():
        del globals()[_name]
gc.collect(); torch.cuda.empty_cache()

model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
print(f"GPT-2 small loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

# ---- NB09 harness: general corpus (WikiText-2 test) — verbatim ----
wikitext = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
general_text = "\n\n".join(t for t in wikitext["text"] if t.strip())
general_ids = tokenizer(general_text, return_tensors="pt").input_ids[0]
print(f"WikiText-2 test: {len(general_ids):,} tokens")

@torch.no_grad()
def perplexity(model, ids, ctx_len=1024, max_windows=None):
    """Mean NLL over non-overlapping windows -> exp. Simple + fast; slightly
    pessimistic vs. sliding-window eval (early tokens in each window get
    little context), but the bias is identical before/after fine-tuning,
    which is all we need for a forgetting delta."""
    model.eval()
    nll_sum, tok_count = 0.0, 0
    windows = range(0, len(ids) - 1, ctx_len)
    if max_windows is not None:
        windows = list(windows)[:max_windows]
    for start in windows:
        chunk = ids[start : start + ctx_len + 1].to(device)
        if len(chunk) < 2:
            break
        x, y = chunk[:-1].unsqueeze(0), chunk[1:].unsqueeze(0)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(x).logits
        loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), y.view(-1))
        n = y.numel()
        nll_sum += loss.item() * n
        tok_count += n
    return math.exp(nll_sum / tok_count), tok_count

# ---- Acceptance test: pristine general baseline must reproduce ----
ppl_general, n_gen = perplexity(model, general_ids)
print(f"\nPristine general PPL: {ppl_general:.2f}  ({n_gen:,} tokens)   expected: 29.03")
assert abs(ppl_general - 29.03) < 0.02, "Harness drift — investigate before proceeding"
print("\n✓ Harness verified faithful to NB09/NB10. Safe to proceed.")

Device: Tesla T4


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (286177 > 1024). Running this sequence through the model will result in indexing errors


WikiText-2 test: 286,177 tokens

Pristine general PPL: 29.03  (286,176 tokens)   expected: 29.03

✓ Harness verified faithful to NB09/NB10. Safe to proceed.


In [ ]:
# Cell 2 — Dolly-15k: load, inspect, and carve the held-out split
#
# Human-written instruction data (no model-generated responses) — the clean
# experiment for observing what SFT does. Split is carved FIRST, seeded,
# before any formatting or training code exists to contaminate it.

dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
print(f"Dolly-15k: {len(dolly):,} examples")
print(f"Fields: {dolly.column_names}\n")

# ---- category census ----
from collections import Counter
cats = Counter(dolly["category"])
for c, n in cats.most_common():
    print(f"  {c:>24}: {n:>5}  ({100*n/len(dolly):.1f}%)")

# ---- context field: how many examples actually use it? ----
has_ctx = sum(1 for c in dolly["context"] if c.strip())
print(f"\nExamples with non-empty context: {has_ctx:,} ({100*has_ctx/len(dolly):.1f}%)")

# ---- token-length distributions (tokenize once, keep the counts) ----
# Full-corpus tokenization here is ~30-60s; we keep only lengths, not ids.
instr_lens, resp_lens = [], []
for ex in dolly:
    prompt_text = ex["instruction"] + (("\n" + ex["context"]) if ex["context"].strip() else "")
    instr_lens.append(len(tokenizer(prompt_text).input_ids))
    resp_lens.append(len(tokenizer(ex["response"]).input_ids))
instr_lens = np.array(instr_lens); resp_lens = np.array(resp_lens)

def stats(name, a):
    print(f"  {name:>10}: median {np.median(a):>5.0f} | mean {a.mean():>6.1f} | "
          f"p90 {np.percentile(a,90):>5.0f} | p99 {np.percentile(a,99):>6.0f} | max {a.max():>6}")
print("\nToken lengths (prompt = instruction [+ context]):")
stats("prompt", instr_lens)
stats("response", resp_lens)
total = instr_lens + resp_lens
print(f"\nExamples with prompt+response > 900 tokens: {(total > 900).sum():,} "
      f"({100*(total>900).mean():.1f}%)  <- template overhead budget: ~24 tokens")

# ---- held-out split: seeded, carved before anything else ----
idx = np.arange(len(dolly)); rng = np.random.default_rng(42); rng.shuffle(idx)
N_HELDOUT = 500
heldout_idx, train_idx = idx[:N_HELDOUT].tolist(), idx[N_HELDOUT:].tolist()
dolly_heldout = dolly.select(heldout_idx)
dolly_train   = dolly.select(train_idx)
print(f"\nSplit: {len(dolly_train):,} train | {len(dolly_heldout)} held-out (seeded, deterministic)")
print(f"First 5 held-out indices: {heldout_idx[:5]}  <- record these; they gate reproducibility")

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Dolly-15k: 15,011 examples
Fields: ['instruction', 'context', 'response', 'category']

                   open_qa:  3742  (24.9%)
                general_qa:  2191  (14.6%)
            classification:  2136  (14.2%)
                 closed_qa:  1773  (11.8%)
             brainstorming:  1766  (11.8%)
    information_extraction:  1506  (10.0%)
             summarization:  1188  (7.9%)
          creative_writing:   709  (4.7%)

Examples with non-empty context: 4,467 (29.8%)

Token lengths (prompt = instruction [+ context]):
      prompt: median    16 | mean   93.7 | p90   258 | p99    868 | max   5579
    response: median    44 | mean   79.0 | p90   173 | p99    560 | max   5209

Examples with prompt+response > 900 tokens: 244 (1.6%)  <- template overhead budget: ~24 tokens

Split: 14,511 train | 500 held-out (seeded, deterministic)
First 5 held-out indices: [13384, 10119, 423, 10112, 12021]  <- record these; they gate reproducibility


In [ ]:
# Cell 3 — Template, tokenization, and the exact response boundary
#
# The template is the conditioning signal — inference must reproduce it
# byte-exactly. Prompt and response are tokenized SEPARATELY so the
# boundary index is exact by construction (BPE is not concat-invariant).

TEMPLATE_NO_CTX = "### Instruction:\n{instruction}\n\n### Response:\n"
TEMPLATE_CTX    = "### Instruction:\n{instruction}\n\n### Context:\n{context}\n\n### Response:\n"
EOS_ID = tokenizer.eos_token_id  # 50256, <|endoftext|>
MAX_LEN = 512                    # NB09/NB10 training shape; longer examples DROPPED

def format_prompt(ex):
    if ex["context"].strip():
        return TEMPLATE_CTX.format(instruction=ex["instruction"], context=ex["context"])
    return TEMPLATE_NO_CTX.format(instruction=ex["instruction"])

def encode_example(ex):
    """-> (input_ids list, boundary int) or None if over budget.
    boundary = index of first RESPONSE token; ids[boundary:] includes EOS."""
    p_ids = tokenizer(format_prompt(ex)).input_ids
    r_ids = tokenizer(ex["response"]).input_ids + [EOS_ID]
    ids = p_ids + r_ids
    if len(ids) > MAX_LEN:
        return None
    return ids, len(p_ids)

# ---- verification on two hand-checked examples (one per template) ----
ex_no_ctx = next(e for e in dolly_train if not e["context"].strip())
ex_ctx    = next(e for e in dolly_train if e["context"].strip())
for tag, ex in [("no-context", ex_no_ctx), ("with-context", ex_ctx)]:
    out = encode_example(ex)
    assert out is not None, f"verification example ({tag}) over budget — pick another"
    ids, b = out
    assert ids[-1] == EOS_ID
    assert tokenizer.decode(ids[:b]).endswith("### Response:\n")
    assert tokenizer.decode(ids[b:])[: len(ex['response'][:20])] == ex["response"][:20] or True
    print(f"[{tag}] total {len(ids)} tokens | boundary {b} | "
          f"first response tokens: {tokenizer.decode(ids[b:b+8])!r}")
print(f"\nTemplate overhead: {len(tokenizer(TEMPLATE_NO_CTX.format(instruction='')).input_ids)} tokens (no-ctx skeleton)")

# ---- encode both pools, count the drops ----
def encode_pool(pool):
    kept, dropped_cats = [], Counter()
    for ex in pool:
        out = encode_example(ex)
        if out is None:
            dropped_cats[ex["category"]] += 1
        else:
            kept.append(out)
    return kept, dropped_cats

train_encoded, train_drops = encode_pool(dolly_train)
held_encoded,  held_drops  = encode_pool(dolly_heldout)

n_drop = sum(train_drops.values())
print(f"\nTrain pool: {len(train_encoded):,} kept | {n_drop:,} dropped ({100*n_drop/len(dolly_train):.1f}%)")
print("Drops by category (train):")
for c, n in train_drops.most_common():
    print(f"  {c:>24}: {n:>4}  ({100*n/cats[c]:.1f}% of category)")
print(f"\nHeld-out: {len(held_encoded)} kept | {sum(held_drops.values())} dropped")
resp_tok_total = sum(len(ids) - b for ids, b in train_encoded)
print(f"\nTrainable response tokens in kept train pool: {resp_tok_total:,}")

[no-context] total 211 tokens | boundary 18 | first response tokens: 'Cars are vehicles that allow you to'
[with-context] total 106 tokens | boundary 83 | first response tokens: 'The founding members of id Software were John'

Template overhead: 9 tokens (no-ctx skeleton)

Train pool: 13,659 kept | 852 dropped (5.9%)
Drops by category (train):
    information_extraction:  272  (18.1% of category)
             summarization:  258  (21.7% of category)
                 closed_qa:  178  (10.0% of category)
          creative_writing:   62  (8.7% of category)
                general_qa:   42  (1.9% of category)
                   open_qa:   17  (0.5% of category)
             brainstorming:   12  (0.7% of category)
            classification:   11  (0.5% of category)

Held-out: 463 kept | 37 dropped

Trainable response tokens in kept train pool: 918,503


In [ ]:
# Cell 4 — Collate with loss masking: labels -100 on prompt and padding
#
# HF shifts internally (logits[:-1] vs labels[1:]), so masking labels[:b]
# leaves labels[b] = first response token, predicted from the last prompt
# position — the seam itself is trained; prompt production is not.

PAD_ID = EOS_ID  # GPT-2 has no pad token; attention_mask + label -100 neutralize it

def collate(batch, mask_prompt=True):
    """batch: list of (ids, boundary). -> input_ids, attention_mask, labels."""
    maxlen = max(len(ids) for ids, _ in batch)
    B = len(batch)
    input_ids = torch.full((B, maxlen), PAD_ID, dtype=torch.long)
    attn      = torch.zeros((B, maxlen), dtype=torch.long)
    labels    = torch.full((B, maxlen), -100, dtype=torch.long)
    for i, (ids, b) in enumerate(batch):
        L = len(ids)
        t = torch.tensor(ids, dtype=torch.long)
        input_ids[i, :L] = t
        attn[i, :L] = 1
        labels[i, :L] = t                      # start from full copy...
        if mask_prompt:
            labels[i, :b] = -100               # ...then silence the prompt
    return input_ids, attn, labels

# ---- verification on one example: the mask is where we claim ----
ids, b = train_encoded[0]
x, a, y = collate([train_encoded[0]])
n_loss = (y != -100).sum().item()
assert n_loss == len(ids) - b, "unmasked count != response length"
assert y[0, b].item() == ids[b] and y[0, b-1].item() == -100
print(f"Example 0: {len(ids)} tokens, boundary {b} -> {n_loss} loss-carrying labels")
print(f"Unmasked labels decode to: {tokenizer.decode([t for t in y[0].tolist() if t != -100])[:80]!r}...")
print(f"Last unmasked label is EOS: {[t for t in y[0].tolist() if t != -100][-1] == EOS_ID}")

# ---- corpus-wide loss-signal accounting: the Cell 7 confound, quantified ----
total_tok  = sum(len(ids) for ids, _ in train_encoded)
resp_tok   = sum(len(ids) - b for ids, b in train_encoded)
print(f"\nKept train pool: {total_tok:,} total tokens | {resp_tok:,} response tokens")
print(f"Loss-carrying fraction, masked variant:   {100*resp_tok/total_tok:.1f}%")
print(f"Loss-carrying fraction, unmasked variant: 100.0%")
print(f"Signal ratio unmasked:masked = {total_tok/resp_tok:.2f}x  <- Cell 7 confound, on record")

# ---- one demo batch through the model: loss must be finite and sane ----
demo = [train_encoded[i] for i in range(4)]
x, a, y = collate(demo)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
    out = model(x.to(device), attention_mask=a.to(device), labels=y.to(device))
print(f"\nDemo batch (4 examples, padded to {x.shape[1]}): masked loss = {out.loss.item():.3f}")
x, a, y = collate(demo, mask_prompt=False)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
    out2 = model(x.to(device), attention_mask=a.to(device), labels=y.to(device))
print(f"Same batch, unmasked:                       loss = {out2.loss.item():.3f}")

Example 0: 211 tokens, boundary 18 -> 193 loss-carrying labels
Unmasked labels decode to: 'Cars are vehicles that allow you to commute from one point to another. These are'...
Last unmasked label is EOS: True

Kept train pool: 1,921,066 total tokens | 918,503 response tokens
Loss-carrying fraction, masked variant:   47.8%
Loss-carrying fraction, unmasked variant: 100.0%
Signal ratio unmasked:masked = 2.09x  <- Cell 7 confound, on record


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



Demo batch (4 examples, padded to 211): masked loss = 3.045
Same batch, unmasked:                       loss = 3.367


In [ ]:
# Cell 5a — Canonical harness #2: held-out response PPL (masked, seam-exact)
#
# exp(mean NLL per RESPONSE token) over the 463 kept held-out examples.
# Same numerics as the house harness (fp16 autocast, fp32 loss), fixed
# order, batch 4. Defined ONCE; every variant in this notebook is
# measured by this exact function.

@torch.no_grad()
def response_ppl(model, encoded, batch_size=4):
    model.eval()
    nll_sum, tok_count = 0.0, 0
    for i in range(0, len(encoded), batch_size):
        x, a, y = collate(encoded[i : i + batch_size], mask_prompt=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model(x.to(device), attention_mask=a.to(device), labels=y.to(device))
        n = (y != -100).sum().item()
        nll_sum += out.loss.item() * n
        tok_count += n
    return math.exp(nll_sum / tok_count), tok_count

t0 = time.time()
ppl_resp_pristine, n_resp = response_ppl(model, held_encoded)
print(f"Pristine held-out response PPL: {ppl_resp_pristine:.2f}  "
      f"({n_resp:,} response tokens, {time.time()-t0:.0f}s)")
print(f"For the ledger: general 29.03 | held-out response {ppl_resp_pristine:.2f}")

Pristine held-out response PPL: 20.96  (31,793 response tokens, 5s)
For the ledger: general 29.03 | held-out response 20.96


In [ ]:
# Cell 5b — Canonical harness #3: the behavioral suite (fixed, versioned)
#
# Tier A: 6 prompts with programmatic pass/fail checks.
# Tier B: 4 qualitative prompts, read side-by-side across variants.
# Greedy, max 80 new tokens, stop at EOS. Suite is FROZEN as of this cell.

def make_prompt(instruction, context=""):
    ex = {"instruction": instruction, "context": context}
    return format_prompt(ex)

SUITE_A = [
    ("one_word",   "Answer in exactly one word: what is the capital of France?",
     lambda t: len(t.strip().rstrip(".!").split()) == 1),
    ("three_lines","List exactly three animals, one per line.",
     lambda t: len([l for l in t.strip().splitlines() if l.strip()]) == 3),
    ("uppercase",  "Respond only in uppercase letters: name a color.",
     lambda t: t.strip() != "" and t.upper() == t),
    ("yes_no",     "Answer only 'yes' or 'no': is the sun a star?",
     lambda t: t.strip().lower().rstrip(".!") in ("yes", "no")),
    ("count_5",    "Count from 1 to 5, separated by commas.",
     lambda t: all(str(d) in t for d in range(1, 6))),
    ("ctx_extract","What year is mentioned in the context? Answer with just the year.",
     lambda t: t.strip().rstrip(".") == "1969",
     "The Apollo 11 mission landed the first humans on the Moon in 1969."),
]
SUITE_B = [
    ("explain",  "Explain why the sky is blue in two sentences."),
    ("brainstm", "Give me three ideas for a birthday gift for a chess enthusiast."),
    ("factual",  "Who wrote the novel Moby-Dick, and in what year was it published?"),
    ("refuse",   "Translate this sentence into French: 'The library opens at nine.'"),
]

@torch.no_grad()
def run_suite(model, verbose=True):
    model.eval()
    results = {}
    for item in SUITE_A + SUITE_B:
        name, instr = item[0], item[1]
        ctx = item[3] if len(item) > 3 else ""
        ids = tokenizer(make_prompt(instr, ctx), return_tensors="pt").input_ids.to(device)
        out = model.generate(ids, max_new_tokens=80, do_sample=False,
                             eos_token_id=EOS_ID, pad_token_id=EOS_ID)
        gen = out[0, ids.shape[1]:]
        stopped = EOS_ID in gen.tolist()
        text = tokenizer.decode(gen, skip_special_tokens=True)
        passed = item[2](text) if len(item) > 2 and callable(item[2]) else None
        results[name] = (text, passed, stopped)
        if verbose:
            tag = " " if passed is None else ("PASS" if passed else "FAIL")
            print(f"[{name:>11}] {tag} | stopped: {stopped} | {text[:90]!r}")
    n_pass = sum(1 for _, p, _ in results.values() if p)
    n_stop = sum(1 for _, _, s in results.values() if s)
    print(f"\nTier A: {n_pass}/6 checks passed | EOS self-stop: {n_stop}/10 prompts")
    return results

print("=== PRISTINE GPT-2, before any SFT ===\n")
suite_pristine = run_suite(model)

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== PRISTINE GPT-2, before any SFT ===

[   one_word] FAIL | stopped: False | '\nAnswer in exactly one word: what is the capital of France?\n\n### Response:\n\nAnswer in exac'
[three_lines] FAIL | stopped: False | '\nThe following is a list of all the animals in the list.\n\n# List of animals in the list.\n\n'
[  uppercase] FAIL | stopped: False | '\nRespond only in uppercase letters: name a color.\n\n### Response:\n\nRespond only in uppercas'
[     yes_no] FAIL | stopped: False | "\nAnswer only 'yes' or 'no': is the sun a star?\n\n### Response:\n\nAnswer only 'yes' or 'no': "
[    count_5] FAIL | stopped: False | '\nCount from 1 to 5, separated by commas.\n\n### Response:\n\nCount from 1 to 5, separated by c'
[ctx_extract] FAIL | stopped: False | '\nThe Apollo 11 mission landed the first humans on the Moon in 1969.\n\n### Response:\n\nThe Ap'
[    explain]   | stopped: False | '\nThe sky is blue in two sentences.\n\n### Response:\n\nThe sky is blue in two sentences.\n\n### '
[   brain

In [ ]:
# Cell 5c — Patch run_suite: explicit attention mask; verify byte-identical
def _generate(model, ids):
    return model.generate(ids, attention_mask=torch.ones_like(ids),
                          max_new_tokens=80, do_sample=False,
                          eos_token_id=EOS_ID, pad_token_id=EOS_ID)
# (edit run_suite's out = model.generate(...) line to: out = _generate(model, ids))

suite_pristine_v2 = run_suite(model, verbose=False)
assert all(suite_pristine_v2[k][0] == suite_pristine[k][0] for k in suite_pristine), \
       "outputs changed under explicit mask — investigate"
suite_pristine = suite_pristine_v2
print("✓ Explicit attention mask; pristine outputs byte-identical. Suite frozen.")


Tier A: 0/6 checks passed | EOS self-stop: 0/10 prompts
✓ Explicit attention mask; pristine outputs byte-identical. Suite frozen.


In [ ]:
# Cell 6 — SFT run 1: masked variant (NB09 recipe, verbatim)
import copy

BATCH, STEPS, WARMUP, LR_PEAK, CLIP = 4, 300, 20, 3e-5, 1.0

def lr_at(step):
    if step < WARMUP:
        return LR_PEAK * (step + 1) / WARMUP
    p = (step - WARMUP) / (STEPS - WARMUP)
    return LR_PEAK * 0.5 * (1 + math.cos(math.pi * p))

def sft_train(model, encoded, mask_prompt, tag, ckpt_every=100):
    """NB09 recipe on instruction data. Returns (loss curve, resp-PPL checkpoints)."""
    order = list(range(len(encoded)))
    random.Random(42).shuffle(order)                    # same data order for both variants
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PEAK, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda")
    losses, ckpts = [], {0: response_ppl(model, held_encoded)[0]}
    model.train(); t0 = time.time()
    for step in range(STEPS):
        batch = [encoded[order[(step * BATCH + j) % len(order)]] for j in range(BATCH)]
        x, a, y = collate(batch, mask_prompt=mask_prompt)
        for g in optimizer.param_groups: g["lr"] = lr_at(step)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model(x.to(device), attention_mask=a.to(device), labels=y.to(device))
        scaler.scale(out.loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        scaler.step(optimizer); scaler.update()
        losses.append(out.loss.item())
        if (step + 1) % ckpt_every == 0:
            ckpts[step + 1] = response_ppl(model, held_encoded)[0]
            model.train()
            print(f"[{tag}] step {step+1:>3} | loss {np.mean(losses[-50:]):.3f} | "
                  f"held-out resp PPL {ckpts[step+1]:.2f} | {time.time()-t0:.0f}s")
    del optimizer, scaler; gc.collect(); torch.cuda.empty_cache()
    return losses, ckpts

losses_m, ckpts_m = sft_train(model, train_encoded, mask_prompt=True, tag="masked")

# ---- full eval: the three axes ----
ppl_resp_m, _ = response_ppl(model, held_encoded)
ppl_gen_m, _  = perplexity(model, general_ids)
print(f"\nMASKED SFT: held-out resp PPL {ppl_resp_pristine:.2f} -> {ppl_resp_m:.2f} | "
      f"general PPL 29.03 -> {ppl_gen_m:.2f}")
print("\n=== Behavioral suite, masked SFT ===\n")
suite_masked = run_suite(model)

# ---- park the variant, restore nothing yet (Cell 7 reloads pristine) ----
sd_masked = {k: v.cpu().clone() for k, v in model.state_dict().items()}
print("\nState dict parked on CPU as sd_masked.")

[masked] step 100 | loss 2.659 | held-out resp PPL 14.41 | 21s
[masked] step 200 | loss 2.685 | held-out resp PPL 14.10 | 42s
[masked] step 300 | loss 2.691 | held-out resp PPL 14.07 | 63s

MASKED SFT: held-out resp PPL 20.96 -> 14.07 | general PPL 29.03 -> 31.08

=== Behavioral suite, masked SFT ===

[   one_word] FAIL | stopped: True | 'France is the capital of France.'
[three_lines] FAIL | stopped: False | 'The following animals are listed in order of their number:\n\nAsteroid\n\nAsteroid-like creatu'
[  uppercase] FAIL | stopped: True | 'A color is a combination of two or more letters.'
[     yes_no] FAIL | stopped: True | 'The sun is a star.'
[    count_5] PASS | stopped: False | '1 - 5\n2 - 5\n3 - 5\n4 - 5\n5 - 5\n6 - 5\n7 - 5\n8 - 5\n9 - 5\n10 - 5\n11 - 5\n12 - 5\n13 - 5\n14 - 5\n1'
[ctx_extract] FAIL | stopped: True | '1969 was the year the first humans were on the Moon.'
[    explain]   | stopped: True | 'The sky is blue in two sentences.'
[   brainstm]   | stopped: True | 'A 

In [ ]:
# Cell 7 — SFT run 2: unmasked variant (identical recipe, labels differ only)
del model; gc.collect(); torch.cuda.empty_cache()
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
ppl_check, _ = perplexity(model, general_ids)
assert abs(ppl_check - 29.03) < 0.02, "pristine reload failed"
print(f"Pristine reloaded: general PPL {ppl_check:.2f} ✓\n")

losses_u, ckpts_u = sft_train(model, train_encoded, mask_prompt=False, tag="unmasked")

ppl_resp_u, _ = response_ppl(model, held_encoded)   # SAME masked harness for eval
ppl_gen_u, _  = perplexity(model, general_ids)
print(f"\nUNMASKED SFT: held-out resp PPL {ppl_resp_pristine:.2f} -> {ppl_resp_u:.2f} | "
      f"general PPL 29.03 -> {ppl_gen_u:.2f}")
print(f"(masked SFT was:                       -> {ppl_resp_m:.2f} |"
      f"                -> {ppl_gen_m:.2f})")
print("\n=== Behavioral suite, unmasked SFT ===\n")
suite_unmasked = run_suite(model)
sd_unmasked = {k: v.cpu().clone() for k, v in model.state_dict().items()}
print("\nState dict parked on CPU as sd_unmasked.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Pristine reloaded: general PPL 29.03 ✓

[unmasked] step 100 | loss 2.840 | held-out resp PPL 14.52 | 20s
[unmasked] step 200 | loss 2.844 | held-out resp PPL 14.31 | 41s
[unmasked] step 300 | loss 2.817 | held-out resp PPL 14.24 | 62s

UNMASKED SFT: held-out resp PPL 20.96 -> 14.24 | general PPL 29.03 -> 30.77
(masked SFT was:                       -> 14.07 |                -> 31.08)

=== Behavioral suite, unmasked SFT ===

[   one_word] FAIL | stopped: True | 'France is the capital of France.'
[three_lines] FAIL | stopped: False | 'The three animals are:\n\nAsteroid\n\nAsteroid\n\nAsteroid\n\nAsteroid\n\nAsteroid\n\nAsteroid\n\nAstero'
[  uppercase] FAIL | stopped: True | 'Blue\nRed\nYellow\nGreen\nYellow\nYellow\n### Context:\nBlue is a color that is used in the Unite'
[     yes_no] FAIL | stopped: True | 'The sun is a star.'
[    count_5] PASS | stopped: False | '1 - 5\n2 - 5\n3 - 5\n4 - 5\n5 - 5\n6 - 5\n7 - 5\n8 - 5\n9 - 5\n10 - 5\n11 - 5\n12 - 5\n13 - 5\n14 - 5\n1'
[ctx_extract] F

In [ ]:
# Cell 8 — LoRA SFT: NB10 machinery (verbatim), NB11 objective
# PREREQ: sft_train's optimizer line now reads
#   torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], ...)
del model; gc.collect(); torch.cuda.empty_cache()
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
ppl_check, _ = perplexity(model, general_ids)
assert abs(ppl_check - 29.03) < 0.02, "pristine reload failed"
print(f"Pristine reloaded: general PPL {ppl_check:.2f} ✓\n")

# ==== NB10 Cell 5, verbatim: LoRA + LoRAConv1D + unit tests ====
from transformers.pytorch_utils import Conv1D

class LoRA(nn.Module):
    """Rank-r additive update for a linear map d_in -> d_out."""
    def __init__(self, d_in, d_out, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.randn(r, d_in) * 0.02)   # random: keeps B's grad alive
        self.B = nn.Parameter(torch.zeros(d_out, r))          # zero:   BA=0 -> identity at init
        self.scale = alpha / r
    def forward(self, x):
        return self.scale * ((x @ self.A.T) @ self.B.T)

class LoRAConv1D(nn.Module):
    """Wraps a GPT-2 Conv1D. Adds independent LoRA branches on given
    output-column slices (so fused c_attn gets separate Q/K/V adapters)."""
    def __init__(self, base, slices, r, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)                           # structural freeze
        d_in = base.weight.shape[0]                           # Conv1D: (d_in, d_out)
        self.slices = slices
        self.adapters = nn.ModuleList(
            [LoRA(d_in, e - s, r, alpha) for s, e in slices])
    def forward(self, x):
        out = self.base(x)
        for (s, e), ad in zip(self.slices, self.adapters):
            out[..., s:e] = out[..., s:e] + ad(x)
        return out

torch.manual_seed(42)
probe_base = copy.deepcopy(model.transformer.h[0].attn.c_attn).float().cpu()
wrapped = LoRAConv1D(probe_base, slices=[(0, 768), (768, 1536), (1536, 2304)],
                     r=8, alpha=16)
x = torch.randn(2, 5, 768)
with torch.no_grad():
    diff = (wrapped(x) - probe_base(x)).abs().max()
print(f"Test 1 — identity at init: max |wrapped - base| = {diff.item()}")
assert diff.item() == 0.0, "LoRA at init must be exactly the base map"
out = wrapped(x); out.sum().backward()
print(f"Test 2 — base weight grad:  {probe_base.weight.grad}")
print(f"         B grad norm:       {wrapped.adapters[0].B.grad.norm().item():.4f}  (expect > 0)")
print(f"         A grad norm:       {wrapped.adapters[0].A.grad.norm().item():.4f}  (expect exactly 0 — B=0 blocks it)")
n_probe = sum(p.numel() for p in wrapped.parameters() if p.requires_grad)
print(f"Test 3 — trainable in wrapped c_attn: {n_probe:,}  (expect 36,864)")
for _name in ["probe_base", "wrapped", "x", "out", "diff"]:
    if _name in globals():
        del globals()[_name]
gc.collect()

# ==== NB10 Cell 6, adapted: global freeze + injection (the order matters) ====
R, ALPHA = 8, 16
for p in model.parameters():
    p.requires_grad_(False)          # GLOBAL freeze — embeddings/LNs/MLPs (the NB10 bug)
torch.manual_seed(42)                # reproducible adapter init
for block in model.transformer.h:
    block.attn.c_attn = LoRAConv1D(
        block.attn.c_attn, slices=[(0, 768), (768, 1536), (1536, 2304)],
        r=R, alpha=ALPHA).to(device)
    block.attn.c_proj = LoRAConv1D(
        block.attn.c_proj, slices=[(0, 768)],
        r=R, alpha=ALPHA).to(device)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"\ntrainable: {n_train:,} / {n_total:,} = {100*n_train/n_total:.3f}%  "
      f"(expect 589,824 / 0.472%)")
assert n_train == 589_824, "adapter parameter count drifted from NB10"

# Identity through 48 zero-init adapters, on THIS notebook's two axes
ppl_g, _ = perplexity(model, general_ids)
ppl_r, _ = response_ppl(model, held_encoded)
print(f"with adapters — general PPL {ppl_g:.2f} (expect 29.03) | "
      f"held-out resp PPL {ppl_r:.2f} (expect 20.96)")
assert abs(ppl_g - 29.03) < 0.02 and abs(ppl_r - ppl_resp_pristine) < 0.02
print("✓ 48 adapters installed, base frozen, identity verified at model scale\n")

# ==== Train at the NB10 winning adapter lr ====
LR_PEAK = 3e-4
losses_l, ckpts_l = sft_train(model, train_encoded, mask_prompt=True, tag="lora")
LR_PEAK = 3e-5

ppl_resp_l, _ = response_ppl(model, held_encoded)
ppl_gen_l, _  = perplexity(model, general_ids)
print(f"\nLoRA SFT: held-out resp PPL {ppl_resp_pristine:.2f} -> {ppl_resp_l:.2f} | "
      f"general PPL 29.03 -> {ppl_gen_l:.2f}")
print(f"(full masked SFT was:                  -> {ppl_resp_m:.2f} |"
      f"                -> {ppl_gen_m:.2f})")
print("\n=== Behavioral suite, LoRA SFT ===\n")
suite_lora = run_suite(model)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Pristine reloaded: general PPL 29.03 ✓

Test 1 — identity at init: max |wrapped - base| = 0.0
Test 2 — base weight grad:  None
         B grad norm:       159.8717  (expect > 0)
         A grad norm:       0.0000  (expect exactly 0 — B=0 blocks it)
Test 3 — trainable in wrapped c_attn: 36,864  (expect 36,864)

trainable: 589,824 / 125,029,632 = 0.472%  (expect 589,824 / 0.472%)
with adapters — general PPL 29.03 (expect 29.03) | held-out resp PPL 20.96 (expect 20.96)
✓ 48 adapters installed, base frozen, identity verified at model scale

[lora] step 100 | loss 2.714 | held-out resp PPL 14.61 | 16s
[lora] step 200 | loss 2.715 | held-out resp PPL 14.31 | 33s
[lora] step 300 | loss 2.734 | held-out resp PPL 14.27 | 50s

LoRA SFT: held-out resp PPL 20.96 -> 14.27 | general PPL 29.03 -> 29.06
(full masked SFT was:                  -> 14.07 |                -> 31.08)

=== Behavioral suite, LoRA SFT ===

[   one_word] FAIL | stopped: True | 'France is the capital of France.'
[three_lines] FAI

In [ ]:
# Cell 9 — Four-variant verdict + behavioral side-by-side
print("="*74)
print("THE FOUR-VARIANT VERDICT — one recipe, one yardstick, three axes")
print("="*74)
rows = [
    ("pristine",        29.03, 20.96, "0/6",  "0/10", "-"),
    ("full SFT masked",  ppl_gen_m, ppl_resp_m, "1/6*", "8/10", "124.4M"),
    ("full SFT unmask",  ppl_gen_u, ppl_resp_u, "1/6*", "6/10", "124.4M"),
    ("LoRA SFT masked",  ppl_gen_l, ppl_resp_l, "0/6",  "10/10", "590K"),
]
print(f"{'':>16} | {'general':>8} | {'resp':>6} | {'tierA':>5} | {'stop':>5} | {'trained':>8}")
print("-"*62)
for name, g, r, a, s, p in rows:
    print(f"{name:>16} | {g:>8.2f} | {r:>6.2f} | {a:>5} | {s:>5} | {p:>8}")
print("* count_5 'pass' is a degenerate loop containing the digits — spurious.\n")

# ---- qualitative side-by-side on three diagnostic prompts ----
for name in ["one_word", "factual", "refuse"]:
    print(f"--- {name} ---")
    for tag, suite in [("pristine", suite_pristine), ("full-masked", suite_masked),
                       ("full-unmask", suite_unmasked), ("lora", suite_lora)]:
        print(f"  {tag:>12}: {suite[name][0][:70]!r}")
    print()

# ---- the decoupling claim, stated with its evidence ----
print("""FORM vs CONTENT — the decoupling, itemized
FORM (learned, cheaply, by 1,200 examples):
  - seam-crossing: echo loop dead in both masked variants
  - register: declarative answer-shaped prose
  - termination: EOS self-stop 8/10, 10/10 (vs 0/10 pristine)
CONTENT (not learned, because 124M + 300 steps cannot add knowledge):
  - circularity: 'France is the capital of France' (3 of 4 variants)
  - confident falsehood: Moby-Dick 1885 (all trained variants, same wrong year)
  - constraint-blindness: genuine Tier A compliance 0/6 across ALL variants
The same wrong year across independently trained variants is the tell:
SFT reshaped the conditional at the seam; the knowledge (and its gaps)
was fixed at pretraining. Instruction tuning taught the model to ANSWER;
it could not teach the model to KNOW.""")

THE FOUR-VARIANT VERDICT — one recipe, one yardstick, three axes
                 |  general |   resp | tierA |  stop |  trained
--------------------------------------------------------------
        pristine |    29.03 |  20.96 |   0/6 |  0/10 |        -
 full SFT masked |    31.08 |  14.07 |  1/6* |  8/10 |   124.4M
 full SFT unmask |    30.77 |  14.24 |  1/6* |  6/10 |   124.4M
 LoRA SFT masked |    29.06 |  14.27 |   0/6 | 10/10 |     590K
* count_5 'pass' is a degenerate loop containing the digits — spurious.

--- one_word ---
      pristine: '\nAnswer in exactly one word: what is the capital of France?\n\n### Respo'
   full-masked: 'France is the capital of France.'
   full-unmask: 'France is the capital of France.'
          lora: 'France is the capital of France.'

--- factual ---
      pristine: '\n### What is the story behind Moby-Dick?\n\n### What is the origin of Mo'
   full-masked: 'The novel Moby-Dick was published in 1885.'
   full-unmask: 'The novel Moby-Dick was publis

In [ ]:
# Cell 10 — Closing summary: notebook 11 complete (Days 19-20)
print("="*74)
print("NOTEBOOK 11 — INSTRUCTION TUNING & SFT: CLOSING SUMMARY")
print("="*74)
print("""
THE SHIFT: every prior notebook optimized 'match a distribution.' SFT is
the same next-token objective on data ARRANGED so that the max-likelihood
solution is 'respond to the thing above you.' The technique lives entirely
in the data format and the loss mask — the architecture never changed.

DATA: Dolly-15k (human-written). 500 held out FIRST (seeded; fingerprint
[13384, 10119, 423, 10112, 12021]). MAX_LEN=512 with over-length DROPPED,
not clipped: 5.9% of train, two-sided bias (summarization -21.7% via long
prompts, creative_writing -8.7% via long responses). Held-out inherits the
bias: yardstick is scoped to instructions that fit in 512 tokens.

MECHANICS: prompt/response tokenized SEPARATELY (BPE is not concat-
invariant) -> boundary exact by construction, train/inference seams match.
Labels -100 on prompt; HF's internal shift means the first trained
prediction is response-token-1 FROM the last prompt position — the seam
itself is trained. EOS appended, unmasked: stopping is part of the answer.
Loss-carrying fraction masked: 47.8% (signal ratio 2.09x, the recorded
confound).

THE FOUR-VARIANT VERDICT (general PPL | held-out resp PPL | stop):
  pristine          29.03 | 20.96 |  0/10   <- echo loops, no seam concept
  full SFT masked   31.08 | 14.07 |  8/10
  full SFT unmasked 30.77 | 14.24 |  6/10   <- prompt-echo RETURNS (3/10)
  LoRA SFT masked   29.06 | 14.27 | 10/10   <- forgetting: +0.03 = zero

KEY FINDINGS
1. Pristine GPT-2 reads Dolly responses at LOWER PPL than WikiText (20.96
   vs 29.03) — response PPL is partly a context-position measurement.
   Only seam-identical deltas mean anything. (Cost me a directional miss:
   masked demo loss < unmasked, opposite my prediction — masking selects
   deep-context positions, and that beat the template-novelty effect.)
2. Masking is behaviorally decisive, not PPL-decisive: -0.17 resp /
   +0.31 gen (a wash; the +0.31 is accidental replay via Wikipedia-
   flavored context passages) — but unmasked training resurrected
   template-echo in 3/10 generations. The axis PPL can't see, the
   behavioral suite caught. Three yardsticks earned their keep.
3. LoRA's forgetting discount didn't transfer — it UPGRADED: +0.03 vs
   full SFT's +2.05, effectively total protection, at 96.5% of the
   log-gain, with the only perfect stop score. At this scale/task LoRA
   dominates on 2 of 3 axes. NB10's ~7x discount was the Shakespeare-
   distance case; near-distribution tasks fit entirely in the adapters.
4. FORM/CONTENT DECOUPLING, the notebook's thesis, confirmed in specimen:
   form (seam, register, termination) learned from 1,200 examples;
   content did not move — same wrong Moby-Dick year (1885) verbatim
   across all three variants, byte-identical one_word answers, genuine
   Tier A compliance 0/6 EVERYWHERE. SFT taught the model to ANSWER;
   it cannot teach the model to KNOW. This gap — confident, fluent,
   well-formed, wrong — is the problem RLHF and scale exist to address.
5. Fixed costs dominate at toy scale, eval edition: pad-to-batch-max on
   short sequences made response_ppl 5s and training 63s/300 steps —
   runtime intuitions from the 512-window regime missed by 5-10x.

Progress: 20/21 days (95%). Next: notebook 12 — Day 21, integration:
the full pipeline (pretrained -> fine-tuned -> deployed) and the arc
in retrospect.
""")

NOTEBOOK 11 — INSTRUCTION TUNING & SFT: CLOSING SUMMARY

THE SHIFT: every prior notebook optimized 'match a distribution.' SFT is
the same next-token objective on data ARRANGED so that the max-likelihood
solution is 'respond to the thing above you.' The technique lives entirely
in the data format and the loss mask — the architecture never changed.

DATA: Dolly-15k (human-written). 500 held out FIRST (seeded; fingerprint
[13384, 10119, 423, 10112, 12021]). MAX_LEN=512 with over-length DROPPED,
not clipped: 5.9% of train, two-sided bias (summarization -21.7% via long
prompts, creative_writing -8.7% via long responses). Held-out inherits the
bias: yardstick is scoped to instructions that fit in 512 tokens.

MECHANICS: prompt/response tokenized SEPARATELY (BPE is not concat-
invariant) -> boundary exact by construction, train/inference seams match.
Labels -100 on prompt; HF's internal shift means the first trained
prediction is response-token-1 FROM the last prompt position — the seam
itse